# LeetCode #213 — House Robber II: dãy nhà xếp thành vòng tròn

**Mức độ:** Medium · **Chủ đề:** Array, Dynamic Programming

Skeleton trên LeetCode:

```python
class Solution:
    def rob(self, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Ôn lại **#198 House Robber** (dãy thẳng): `dp[i] = max(dp[i−1], dp[i−2] + nums[i])`.
2. Khó khăn mới: nhà đầu và nhà cuối **kề nhau** → không được cướp cả hai.
3. Kỹ thuật **phá vòng**: chia thành hai bài dãy thẳng — bỏ nhà cuối, hoặc bỏ nhà đầu — rồi lấy max.
4. Chứng minh vì sao hai trường hợp đó **phủ hết** mọi phương án hợp lệ.

# 1. Bài toán là gì?

Các ngôi nhà xếp thành **vòng tròn**: nhà đầu tiên kề nhà cuối cùng. `nums[i]` là số tiền ở nhà `i`. Không được cướp **hai nhà kề nhau**. Trả về số tiền lớn nhất có thể cướp.

**Constraints (theo đề):**

- `1 <= nums.length <= 100`
- `0 <= nums[i] <= 1000`

---

## Ví dụ

```text
nums = [2, 3, 2]     ->  3    (không được lấy cả nhà 0 và nhà 2 vì chúng kề nhau)
nums = [1, 2, 3, 1]  ->  4    (nhà 0 + nhà 2)
nums = [1, 2, 3]     ->  3
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 Áp dụng vào bài này

Vẫn là DP "chọn hay không chọn" của House Robber, nhưng nhà đầu và nhà cuối **kề nhau**. **Phá vòng**: ít nhất một trong hai nhà đó không bị cướp, nên đáp án = `max(` DP trên `nums[:-1]`, DP trên `nums[1:]` `)` — mỗi lần là bài dãy thẳng với hai biến `prev2, prev1`.

# 3. Python cơ bản cần biết

In [3]:
from typing import List
from itertools import combinations
import random, time

nums = [1, 2, 3, 1]
print("bỏ nhà cuối:", nums[:-1], "| bỏ nhà đầu:", nums[1:])
print("hai nhà i, j kề nhau trên vòng n=4 khi (j - i) % 4 in (1, 3):", [(i, j) for i, j in combinations(range(4), 2) if (j - i) % 4 in (1, 3)])

bỏ nhà cuối: [1, 2, 3] | bỏ nhà đầu: [2, 3, 1]
hai nhà i, j kề nhau trên vòng n=4 khi (j - i) % 4 in (1, 3): [(0, 1), (0, 3), (1, 2), (2, 3)]


# 4. Baseline: thử mọi tập nhà

Duyệt mọi tập con, giữ những tập không có hai nhà kề nhau **trên vòng**. `O(2^n · n)` — chỉ dùng để kiểm tra.

In [4]:
def rob_brute(nums: List[int]) -> int:
    n = len(nums)
    best = 0
    for mask in range(1 << n):
        chosen = [i for i in range(n) if mask >> i & 1]
        if n > 1 and any((mask >> i & 1) and (mask >> ((i + 1) % n) & 1) for i in range(n)):
            continue
        best = max(best, sum(nums[i] for i in chosen))
    return best

print(rob_brute([2, 3, 2]), rob_brute([1, 2, 3, 1]), rob_brute([1, 2, 3]), rob_brute([5]))

3 4 3 5


Chú ý trường hợp `n = 1`: `(i + 1) % 1 == i` — nhà duy nhất "kề chính nó" theo công thức, nên phải loại riêng (`n > 1`). Một nhà thì cướp được.

# 5. Ôn lại #198: dãy thẳng

`best(i)` = số tiền lớn nhất từ các nhà `0..i`. Nhà `i` hoặc không cướp (`best(i−1)`), hoặc cướp (`best(i−2) + nums[i]`). Chỉ cần nhớ hai giá trị gần nhất.

In [5]:
def rob_line(nums: List[int]) -> int:
    prev2 = prev1 = 0                    # best(i-2), best(i-1)
    for x in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + x)
    return prev1

print(rob_line([2, 7, 9, 3, 1]), rob_line([]), rob_line([5]))

12 0 5


# 6. Ý tưởng chính: phá vòng

Nhà `0` và nhà `n−1` kề nhau → **ít nhất một trong hai không bị cướp**. Vậy mọi phương án hợp lệ thuộc (ít nhất) một trong hai trường hợp:

- **A:** không cướp nhà `n−1` → bài dãy thẳng trên `nums[0 : n−1]`.
- **B:** không cướp nhà `0` → bài dãy thẳng trên `nums[1 : n]`.

Trong mỗi trường hợp, vòng đã bị "cắt" thành đoạn thẳng, nên ràng buộc vòng tự biến mất. Đáp án = `max(A, B)`. Trường hợp riêng: `n = 1` → `nums[0]` (vì cả hai lát cắt đều rỗng).

```text
      nhà:   0   1   2   3
              \_________/          A: bỏ nhà 3
                  \_________/      B: bỏ nhà 0
```

In [6]:
def rob_circle(nums: List[int]) -> int:
    if len(nums) == 1:
        return nums[0]
    return max(rob_line(nums[:-1]), rob_line(nums[1:]))

print(rob_circle([2, 3, 2]), rob_circle([1, 2, 3, 1]), rob_circle([1, 2, 3]), rob_circle([5]))

3 4 3 5


# 7. Mô phỏng

In [7]:
def trace_line(nums, label):
    prev2 = prev1 = 0
    print(f"{label}: {nums}")
    for i, x in enumerate(nums):
        skip, take = prev1, prev2 + x
        prev2, prev1 = prev1, max(skip, take)
        print(f"   nhà giá {x}: bỏ qua → {skip}, cướp → {take}  ⇒ best = {prev1}")
    return prev1

def trace_circle(nums):
    a = trace_line(nums[:-1], "A (bỏ nhà cuối)")
    b = trace_line(nums[1:], "B (bỏ nhà đầu) ")
    print(f"đáp án = max({a}, {b}) = {max(a, b)}\n")

trace_circle([2, 3, 2])
trace_circle([1, 2, 3, 1])

A (bỏ nhà cuối): [2, 3]
   nhà giá 2: bỏ qua → 0, cướp → 2  ⇒ best = 2
   nhà giá 3: bỏ qua → 2, cướp → 3  ⇒ best = 3
B (bỏ nhà đầu) : [3, 2]
   nhà giá 3: bỏ qua → 0, cướp → 3  ⇒ best = 3
   nhà giá 2: bỏ qua → 3, cướp → 2  ⇒ best = 3
đáp án = max(3, 3) = 3

A (bỏ nhà cuối): [1, 2, 3]
   nhà giá 1: bỏ qua → 0, cướp → 1  ⇒ best = 1
   nhà giá 2: bỏ qua → 1, cướp → 2  ⇒ best = 2
   nhà giá 3: bỏ qua → 2, cướp → 4  ⇒ best = 4
B (bỏ nhà đầu) : [2, 3, 1]
   nhà giá 2: bỏ qua → 0, cướp → 2  ⇒ best = 2
   nhà giá 3: bỏ qua → 2, cướp → 3  ⇒ best = 3
   nhà giá 1: bỏ qua → 3, cướp → 3  ⇒ best = 3
đáp án = max(4, 3) = 4



# 8. Phiên bản đúng format LeetCode

Không cắt lát (tránh tạo bản sao) — truyền khoảng chỉ số.

In [8]:
class Solution:
    def rob(self, nums: List[int]) -> int:
        n = len(nums)
        if n == 1:
            return nums[0]

        def line(lo: int, hi: int) -> int:      # nhà lo..hi-1
            prev2 = prev1 = 0
            for i in range(lo, hi):
                prev2, prev1 = prev1, max(prev1, prev2 + nums[i])
            return prev1

        return max(line(0, n - 1), line(1, n))

In [9]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.rob([2, 3, 2]), 3)
check("Ví dụ 2", sol.rob([1, 2, 3, 1]), 4)
check("Ví dụ 3", sol.rob([1, 2, 3]), 3)
check("Một nhà", sol.rob([7]), 7)
check("Hai nhà", sol.rob([4, 9]), 9)
check("Toàn số 0", sol.rob([0, 0, 0]), 0)

random.seed(213)
for _ in range(3000):
    nums = [random.randint(0, 20) for _ in range(random.randint(1, 12))]
    assert sol.rob(nums) == rob_brute(nums) == rob_circle(nums), nums
print("3000 test ngẫu nhiên: khớp brute force duyệt mọi tập con")

big = [random.randint(0, 1000) for _ in range(100)]
t0 = time.time()
for _ in range(1000):
    sol.rob(big)
t1 = time.time()
print(f"n=100 (giới hạn đề), 1000 lần gọi: {t1 - t0:.3f}s")

Ví dụ 1          PASS  3
Ví dụ 2          PASS  4
Ví dụ 3          PASS  3
Một nhà          PASS  7
Hai nhà          PASS  9
Toàn số 0        PASS  0


3000 test ngẫu nhiên: khớp brute force duyệt mọi tập con
n=100 (giới hạn đề), 1000 lần gọi: 0.011s


# 9. Vì sao đúng? Độ phức tạp?

- **Mọi phương án của A hoặc B đều hợp lệ trên vòng:** A không chứa nhà `n−1` nên cặp kề duy nhất "vòng qua" (`n−1`, `0`) không thể bị vi phạm; các cặp kề còn lại nằm trong đoạn thẳng đã được DP xử lý. Tương tự cho B.
- **Mọi phương án hợp lệ trên vòng thuộc A hoặc B:** nó không chứa cả `0` lẫn `n−1`, nên hoặc thiếu `n−1` (thuộc A) hoặc thiếu `0` (thuộc B).

Vậy `max(A, B)` = tối ưu trên vòng. (Phương án thiếu cả hai nhà thuộc cả A và B — không sao, ta lấy max chứ không cộng.)

```text
Time:  O(n)    (hai lần DP dãy thẳng)
Space: O(1)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Giải như dãy thẳng, quên ràng buộc vòng

In [10]:
print("[2, 3, 2] — sai:", rob_line([2, 3, 2]), "| đúng:", sol.rob([2, 3, 2]))

[2, 3, 2] — sai: 4 | đúng: 3


## Lỗi 2: Quên trường hợp một nhà

In [11]:
def wrong_single(nums):
    return max(rob_line(nums[:-1]), rob_line(nums[1:]))

print("[7] — sai:", wrong_single([7]), "| đúng:", sol.rob([7]))

[7] — sai: 0 | đúng: 7


## Lỗi 3: Giải dãy thẳng rồi "sửa hậu kỳ"

Ý tưởng sai hay gặp: lấy phương án tối ưu của dãy thẳng; nếu nó cướp **cả** nhà đầu lẫn nhà cuối thì bỏ nhà rẻ hơn trong hai nhà đó.

In [12]:
def line_choice(arr):
    """DP dãy thẳng có truy vết: trả về danh sách chỉ số được chọn."""
    m = len(arr)
    dp = [0] * (m + 1)
    for k in range(1, m + 1):
        dp[k] = max(dp[k - 1], (dp[k - 2] if k >= 2 else 0) + arr[k - 1])
    idx, k = [], m
    while k >= 1:
        if dp[k] == dp[k - 1]:
            k -= 1
        else:
            idx.append(k - 1); k -= 2
    return idx[::-1]

def wrong_post_fix(nums):
    idx = line_choice(nums)
    n = len(nums)
    if n > 1 and 0 in idx and n - 1 in idx:
        idx.remove(0 if nums[0] <= nums[-1] else n - 1)      # SAI: chỉ bỏ một nhà, không tối ưu lại
    return sum(nums[i] for i in idx)

random.seed(3)
for _ in range(10000):
    nums = [random.randint(0, 9) for _ in range(random.randint(3, 6))]
    if wrong_post_fix(nums) != rob_brute(nums):
        print("Phản ví dụ:", nums, "→ phương án dãy thẳng:", line_choice(nums), "| sai:", wrong_post_fix(nums), "| đúng:", rob_brute(nums))
        break

Phản ví dụ: [9, 8, 2, 5] → phương án dãy thẳng: [0, 3] | sai: 9 | đúng: 13


Sửa "hậu kỳ" một đáp án DP thường sai: phương án tối ưu trên vòng có thể **hoàn toàn khác** phương án tối ưu trên dãy thẳng, không chỉ khác ở một nhà. Phải chia trường hợp **trước** khi DP.

## Lỗi 4: Lấy `nums[1:-1]` (bỏ cả hai đầu)

In [13]:
def wrong_drop_both(nums):
    if len(nums) <= 2:
        return max(nums)
    return rob_line(nums[1:-1])

print("[1, 2, 3, 1] — sai:", wrong_drop_both([1, 2, 3, 1]), "| đúng:", sol.rob([1, 2, 3, 1]))

[1, 2, 3, 1] — sai: 3 | đúng: 4


Bỏ cả hai đầu thì mất các phương án dùng nhà `0` (hoặc nhà `n−1`). Chỉ cần bỏ **một** đầu mỗi lần.

# 11. Mẹo nhận diện pattern

- **Ràng buộc vòng tròn** (phần tử đầu kề phần tử cuối) → **phá vòng**: cố định trạng thái của một phần tử (lấy / không lấy) rồi giải trên dãy thẳng.
- **Không chọn hai phần tử kề nhau** → DP `max(bỏ, lấy + dp[i−2])`.
- Kỹ thuật phá vòng cũng gặp ở: mảng con tổng lớn nhất trên vòng (#918), sơn nhà trên vòng.

Bài liên quan: **#198 House Robber**, **#337 House Robber III** (trên cây), **#740 Delete and Earn**, **#918 Maximum Sum Circular Subarray**, **#1388 Pizza With 3n Slices** (Hard), **#2560 House Robber IV**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `nums = [6, 1, 2, 7, 3, 5]`, tính kết quả A (bỏ nhà cuối) và B (bỏ nhà đầu). Đáp án là bao nhiêu?

## Bài 2 — Điền chỗ trống

In [14]:
def exercise_2_rob(nums: List[int]) -> int:
    if len(nums) == 1:
        return nums[0]
    def line(arr):
        prev2 = prev1 = 0
        for x in arr:
            # TODO
            pass
        return prev1
    return max(line(nums[:-1]), line(nums[1:]))


# assert exercise_2_rob([2, 3, 2]) == 3
# assert exercise_2_rob([1, 2, 3, 1]) == 4
# assert exercise_2_rob([7]) == 7

## Bài 3 — Biến thể: trả về **danh sách chỉ số** nhà bị cướp (một phương án tối ưu)

In [15]:
def exercise_3_rob_indices(nums: List[int]) -> List[int]:
    # TODO
    pass


def valid_circle_choice(nums, idx):
    n, s = len(nums), set(idx)
    return len(s) == len(idx) and all(0 <= i < n for i in idx) and (n == 1 or all(not (i in s and (i + 1) % n in s) for i in range(n)))

# assert sum([1, 2, 3, 1][i] for i in exercise_3_rob_indices([1, 2, 3, 1])) == 4 and valid_circle_choice([1, 2, 3, 1], exercise_3_rob_indices([1, 2, 3, 1]))
# assert exercise_3_rob_indices([2, 3, 2]) == [1]
# assert exercise_3_rob_indices([7]) == [0]

## Bài 4 — Biến thể: Maximum Sum Circular Subarray (LeetCode #918)

Mảng vòng tròn, tìm tổng lớn nhất của một mảng con khác rỗng (có thể "vòng qua" cuối về đầu). Gợi ý: đáp án = `max(Kadane max, tổng − Kadane min)`, trừ khi mọi số âm.

In [16]:
def exercise_4_max_subarray_sum_circular(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_max_subarray_sum_circular([1, -2, 3, -2]) == 3
# assert exercise_4_max_subarray_sum_circular([5, -3, 5]) == 10
# assert exercise_4_max_subarray_sum_circular([-3, -2, -3]) == -2

## Bài 5 — Mini challenge: House Robber III (LeetCode #337)

Các nhà là nút của **cây nhị phân**; không được cướp hai nhà có quan hệ cha–con. Gợi ý: mỗi nút trả về cặp `(cướp nút này, không cướp nút này)`.

In [17]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right


def build_tree(vals):
    """Dựng cây từ danh sách theo tầng kiểu LeetCode (None = không có nút)."""
    if not vals:
        return None
    it = iter(vals)
    root = TreeNode(next(it))
    q = [root]
    for node in q:
        for side in ("left", "right"):
            v = next(it, None)
            if v is not None:
                child = TreeNode(v)
                setattr(node, side, child)
                q.append(child)
    return root


def exercise_5_rob_tree(root: TreeNode) -> int:
    # TODO
    pass


# assert exercise_5_rob_tree(build_tree([3, 2, 3, None, 3, None, 1])) == 7
# assert exercise_5_rob_tree(build_tree([3, 4, 5, 1, 3, None, 1])) == 9
# assert exercise_5_rob_tree(build_tree([5])) == 5

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_circle([6, 1, 2, 7, 3, 5])` sau khi tự làm.

**Bài 3**
- Mức 1: làm DP dãy thẳng **có lưu mảng** `dp`, truy vết ngược: nếu `dp[i] == dp[i−1]` thì nhà `i` không được chọn, ngược lại chọn và nhảy về `i−2`.
- Mức 2: làm cho cả hai lát cắt A và B, chọn lát cắt có tổng lớn hơn, nhớ cộng độ lệch chỉ số (B bắt đầu từ 1).

**Bài 4**
- Mức 1: mảng con vòng qua = tổng toàn bộ trừ đi một mảng con **không** vòng qua (phần bị bỏ) → bỏ phần có tổng nhỏ nhất.
- Mức 2: nếu Kadane max < 0 (mọi số âm) thì trả Kadane max (không được bỏ cả mảng).

**Bài 5**
- Mức 1: `rob = node.val + L.skip + R.skip`; `skip = max(L) + max(R)`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [18]:
trace_circle([6, 1, 2, 7, 3, 5])

A (bỏ nhà cuối): [6, 1, 2, 7, 3]
   nhà giá 6: bỏ qua → 0, cướp → 6  ⇒ best = 6
   nhà giá 1: bỏ qua → 6, cướp → 1  ⇒ best = 6
   nhà giá 2: bỏ qua → 6, cướp → 8  ⇒ best = 8
   nhà giá 7: bỏ qua → 8, cướp → 13  ⇒ best = 13
   nhà giá 3: bỏ qua → 13, cướp → 11  ⇒ best = 13
B (bỏ nhà đầu) : [1, 2, 7, 3, 5]
   nhà giá 1: bỏ qua → 0, cướp → 1  ⇒ best = 1
   nhà giá 2: bỏ qua → 1, cướp → 2  ⇒ best = 2
   nhà giá 7: bỏ qua → 2, cướp → 8  ⇒ best = 8
   nhà giá 3: bỏ qua → 8, cướp → 5  ⇒ best = 8
   nhà giá 5: bỏ qua → 8, cướp → 13  ⇒ best = 13
đáp án = max(13, 13) = 13



## Đáp án Bài 2

In [19]:
def answer_2_rob(nums: List[int]) -> int:
    if len(nums) == 1:
        return nums[0]
    def line(arr):
        prev2 = prev1 = 0
        for x in arr:
            prev2, prev1 = prev1, max(prev1, prev2 + x)
        return prev1
    return max(line(nums[:-1]), line(nums[1:]))

assert answer_2_rob([2, 3, 2]) == 3
assert answer_2_rob([1, 2, 3, 1]) == 4
assert answer_2_rob([7]) == 7
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [20]:
def answer_3_rob_indices(nums: List[int]) -> List[int]:
    n = len(nums)
    if n == 1:
        return [0]

    def line_with_indices(lo, hi):
        arr = nums[lo:hi]
        m = len(arr)
        dp = [0] * (m + 1)                      # dp[k] = tốt nhất với k nhà đầu của arr
        for k in range(1, m + 1):
            dp[k] = max(dp[k - 1], (dp[k - 2] if k >= 2 else 0) + arr[k - 1])
        idx, k = [], m
        while k >= 1:
            if dp[k] == dp[k - 1]:
                k -= 1
            else:
                idx.append(lo + k - 1)
                k -= 2
        return dp[m], idx[::-1]

    a = line_with_indices(0, n - 1)
    b = line_with_indices(1, n)
    return (a if a[0] >= b[0] else b)[1]

assert sum([1, 2, 3, 1][i] for i in answer_3_rob_indices([1, 2, 3, 1])) == 4 and valid_circle_choice([1, 2, 3, 1], answer_3_rob_indices([1, 2, 3, 1]))
assert answer_3_rob_indices([2, 3, 2]) == [1]
assert answer_3_rob_indices([7]) == [0]
for _ in range(2000):
    nums = [random.randint(0, 20) for _ in range(random.randint(1, 10))]
    idx = answer_3_rob_indices(nums)
    assert valid_circle_choice(nums, idx) and sum(nums[i] for i in idx) == rob_brute(nums), (nums, idx)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [21]:
def answer_4_max_subarray_sum_circular(nums: List[int]) -> int:
    cur_max = cur_min = 0
    best_max, best_min = nums[0], nums[0]
    for x in nums:
        cur_max = max(cur_max + x, x)
        best_max = max(best_max, cur_max)
        cur_min = min(cur_min + x, x)
        best_min = min(best_min, cur_min)
    if best_max < 0:
        return best_max
    return max(best_max, sum(nums) - best_min)

def circular_brute(nums):
    n = len(nums)
    return max(sum(nums[(i + k) % n] for k in range(length)) for i in range(n) for length in range(1, n + 1))

assert answer_4_max_subarray_sum_circular([1, -2, 3, -2]) == 3
assert answer_4_max_subarray_sum_circular([5, -3, 5]) == 10
assert answer_4_max_subarray_sum_circular([-3, -2, -3]) == -2
for _ in range(2000):
    nums = [random.randint(-10, 10) for _ in range(random.randint(1, 8))]
    assert answer_4_max_subarray_sum_circular(nums) == circular_brute(nums), nums
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [22]:
def answer_5_rob_tree(root: TreeNode) -> int:
    def go(node):
        if node is None:
            return 0, 0                          # (cướp, không cướp)
        lr, ls = go(node.left)
        rr, rs = go(node.right)
        return node.val + ls + rs, max(lr, ls) + max(rr, rs)
    return max(go(root))

def rob_tree_brute(root):
    nodes, parent = [], {}
    q = [root]
    for node in q:
        nodes.append(node)
        for ch in (node.left, node.right):
            if ch:
                parent[id(ch)] = node
                q.append(ch)
    best = 0
    for mask in range(1 << len(nodes)):
        chosen = {id(nodes[i]) for i in range(len(nodes)) if mask >> i & 1}
        if any(id(parent[c]) in chosen for c in chosen if c in parent):
            continue
        best = max(best, sum(nodes[i].val for i in range(len(nodes)) if mask >> i & 1))
    return best

assert answer_5_rob_tree(build_tree([3, 2, 3, None, 3, None, 1])) == 7
assert answer_5_rob_tree(build_tree([3, 4, 5, 1, 3, None, 1])) == 9
assert answer_5_rob_tree(build_tree([5])) == 5
for _ in range(500):
    vals = [random.randint(0, 9)] + [random.choice([None, random.randint(0, 9)]) for _ in range(random.randint(0, 8))]
    root = build_tree(vals)
    assert answer_5_rob_tree(root) == rob_tree_brute(root), vals
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Vòng tròn ⟹ nhà đầu và nhà cuối không thể cùng bị cướp ⟹ **phá vòng**: đáp án = `max(rob_line(nums[:-1]), rob_line(nums[1:]))`, riêng `n = 1` trả `nums[0]`. Mỗi `rob_line` là DP hai biến `prev2, prev1 = prev1, max(prev1, prev2 + x)`. `O(n)` thời gian, `O(1)` bộ nhớ.

---

## Checklist trước khi nộp

- [ ] Xử lý riêng `n = 1`?
- [ ] Bỏ **một** đầu mỗi lần (không phải cả hai)?
- [ ] Lấy `max` của hai trường hợp?
- [ ] DP dãy thẳng đúng công thức `max(bỏ, lấy + dp[i−2])`?